# PoC: RAG Evaluation com RAGAs

Vamos avaliar sistematicamente a qualidade do nosso sistema RAG.

## Pipeline de Avaliacao

```
Dataset de teste (perguntas + respostas esperadas)
    |
RAG Pipeline → gera respostas + contextos
    |
RAGAs → calcula metricas
    |
Analise → identifica areas de melhoria
```

**Prerequisito:** `docker compose up -d` e `docker exec ollama ollama pull llama3.2`

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import pandas as pd
import numpy as np
import ollama
import httpx
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from src.utils.chunking import recursive_chunk

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
client = QdrantClient(host='localhost', port=6333)

try:
    r = httpx.get('http://localhost:11434/api/tags')
    modelos = [m['name'] for m in r.json().get('models', [])]
    LLM = 'llama3.2' if any('llama3.2' in m for m in modelos) else (modelos[0] if modelos else None)
    print(f'LLM: {LLM}')
except:
    LLM = None
    print('Ollama offline')

In [ ]:
# Dataset de avaliacao manual
# Em producao: criar com LLM ou anotacao humana

eval_dataset = [
    {
        'question': 'O que e HNSW e quais sao seus parametros principais?',
        'ground_truth': 'HNSW e Hierarchical Navigable Small World, um algoritmo de indexacao para busca aproximada. Os parametros principais sao m (conexoes por no, default 16) e ef_construct (candidatos na construcao, default 100).',
    },
    {
        'question': 'Qual e a diferenca entre float32 e int8 para armazenamento de embeddings?',
        'ground_truth': 'float32 usa 4 bytes por dimensao com precisao total. int8 usa 1 byte por dimensao (4x menos memoria) com ~97-99% de qualidade preservada. int8 e usado na quantizacao scalar do Qdrant.',
    },
    {
        'question': 'O que e RAG e como ele previne alucinacoes?',
        'ground_truth': 'RAG (Retrieval-Augmented Generation) busca documentos relevantes e usa como contexto para o LLM. Previne alucinacoes porque o LLM responde baseado em fatos reais dos documentos, nao apenas em conhecimento parametrico.',
    },
    {
        'question': 'Quais sao os tipos de chunking para RAG?',
        'ground_truth': 'Os principais tipos sao: fixed-size (tamanho fixo em caracteres), recursive (separa em separadores hierarquicos), semantic (usa embedding para detectar mudancas de topico) e document-aware (respeita estrutura do documento).',
    },
]

print(f'Dataset de avaliacao: {len(eval_dataset)} perguntas')
for i, d in enumerate(eval_dataset):
    print(f'  {i+1}. {d["question"][:60]}...')

In [ ]:
# Setup: indexar documentos se necessario
COLLECTION = 'eval_rag'

if COLLECTION not in [c.name for c in client.get_collections().collections]:
    docs_dir = Path('../../data/sample_docs')
    all_chunks, all_payloads = [], []
    for p in docs_dir.glob('*.md'):
        text = p.read_text(encoding='utf-8')
        for c in recursive_chunk(text, chunk_size=400, overlap=50):
            all_chunks.append(c.text)
            all_payloads.append({'text': c.text, 'source': p.name})
    
    embs = embed_model.encode(all_chunks, normalize_embeddings=True, show_progress_bar=True)
    if client.collection_exists(COLLECTION):
        client.delete_collection(COLLECTION)
    client.create_collection(COLLECTION, vectors_config=VectorParams(size=384, distance=Distance.COSINE))
    points = [PointStruct(id=i, vector=embs[i].tolist(), payload=all_payloads[i]) for i in range(len(all_chunks))]
    client.upsert(COLLECTION, points=points)
    print(f'{len(all_chunks)} chunks indexados')

def run_rag(question, top_k=5):
    q_vec = embed_model.encode(question, normalize_embeddings=True)
    results = client.query_points(COLLECTION, query=q_vec.tolist(), limit=top_k, with_payload=True).points
    contexts = [r.payload.get('text', '') for r in results]
    
    context_str = '\n\n'.join(contexts)
    prompt = f'Responda APENAS com base no contexto abaixo.\n\nContexto:\n{context_str}\n\nPergunta: {question}\n\nResposta:'
    
    if LLM:
        response = ollama.chat(model=LLM, messages=[{'role': 'user', 'content': prompt}])
        answer = response['message']['content']
    else:
        answer = f'[LLM offline] Contexto: {context_str[:200]}'
    
    return answer, contexts

print('Setup pronto!')

In [ ]:
# Gerar respostas para o dataset de avaliacao
print('Gerando respostas do sistema RAG...')

resultados = []
for item in eval_dataset:
    answer, contexts = run_rag(item['question'])
    resultados.append({
        'question': item['question'],
        'answer': answer,
        'contexts': contexts,
        'ground_truth': item['ground_truth'],
    })
    print(f'  Q: {item["question"][:50]}...')
    print(f'  A: {answer[:100]}...\n')

In [ ]:
# Avaliacao com RAGAs
try:
    from ragas import evaluate
    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    )
    from datasets import Dataset
    from langchain_ollama import ChatOllama
    from langchain_ollama.embeddings import OllamaEmbeddings
    
    # Preparar dataset para RAGAs
    ragas_dataset = Dataset.from_list(resultados)
    
    # Configurar LLM e embeddings locais
    llm = ChatOllama(model=LLM or 'llama3.2', temperature=0)
    emb = OllamaEmbeddings(model='nomic-embed-text')
    
    print('Executando RAGAs evaluation...')
    scores = evaluate(
        dataset=ragas_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=llm,
        embeddings=emb,
    )
    
    print('\nRAGAs Scores:')
    df = scores.to_pandas()
    print(df[['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']].describe())

except ImportError as e:
    print(f'RAGAs nao disponivel: {e}')
    print('Execute: uv add ragas datasets langchain-ollama')
    
    # Avaliacao manual simplificada
    print('\nAvaliacao manual simplificada:')
    
    def naive_faithfulness(answer, contexts):
        context_text = ' '.join(contexts).lower()
        words = answer.lower().split()
        content_words = [w for w in words if len(w) > 4]
        found = sum(1 for w in content_words if w in context_text)
        return found / len(content_words) if content_words else 0
    
    for r in resultados:
        faith = naive_faithfulness(r['answer'], r['contexts'])
        print(f'  Q: {r["question"][:50]}...')
        print(f'  Faithfulness (aprox): {faith:.2f}')

In [ ]:
# Visualizar resultados
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Scores de demonstracao (substitua pelos scores reais do RAGAs)
metrics_example = {
    'Faithfulness': 0.82,
    'Answer Relevancy': 0.78,
    'Context Precision': 0.71,
    'Context Recall': 0.85,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Radar chart
categories = list(metrics_example.keys())
values = list(metrics_example.values())
N = len(categories)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
values_plot = values + values[:1]

ax = axes[0]
ax = plt.subplot(121, polar=True)
ax.plot(angles, values_plot, 'o-', linewidth=2, color='#3498db')
ax.fill(angles, values_plot, alpha=0.25, color='#3498db')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1)
ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, linewidth=1)
ax.set_title('RAGAs Score (exemplo)', fontsize=12, fontweight='bold', pad=20)

# Bar chart
ax2 = axes[1]
bars = ax2.barh(list(metrics_example.keys()), list(metrics_example.values()),
               color=['#2ecc71' if v >= 0.8 else '#f39c12' if v >= 0.6 else '#e74c3c'
                      for v in metrics_example.values()])
ax2.axvline(x=0.8, color='green', linestyle='--', alpha=0.7, label='Target 80%')
ax2.axvline(x=0.6, color='orange', linestyle='--', alpha=0.7, label='Min 60%')
ax2.set_xlim(0, 1)
ax2.set_title('RAGAs Metricas por Dimensao', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
for bar, val in zip(bars, metrics_example.values()):
    ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontweight='bold')

plt.suptitle('Avaliacao do Sistema RAG (RAGAs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nInterpretacao:')
for metrica, score in metrics_example.items():
    status = 'OTIMO' if score >= 0.8 else 'ADEQUADO' if score >= 0.6 else 'PRECISA MELHORAR'
    print(f'  {metrica}: {score:.2f} → {status}')

## Como melhorar os scores?

| Score baixo em... | Possiveis causas | Solucoes |
|-------------------|-----------------|----------|
| Faithfulness | LLM alucina alem do contexto | Prompt mais restritivo, contexto menor |
| Answer Relevancy | Recupera docs errados | Melhorar embedding model, query rewriting |
| Context Precision | Top-k recupera ruido | Re-ranking, score threshold |
| Context Recall | Chunks importantes faltando | Maior k, melhor chunking, hybrid search |

## Proximo
- [06 Evaluation — RAGAs em profundidade](../../06_evaluation/01_ragas_metrics.ipynb)